# Google Search Console Anomaly Detection with Local Outlier Factor (LOF)
## Density-Based Anomaly Detection for Traffic Patterns

This notebook uses Local Outlier Factor (LOF), a density-based anomaly detection algorithm that excels at finding local anomalies.

**Why LOF?**
- Detects anomalies based on local density deviation
- Better at finding subtle, localized anomalies than Isolation Forest
- Considers the local neighborhood of each point
- Great for data with varying density patterns
- Handles clusters of normal behavior well

**How LOF Works:**
- Compares the local density of a point with its neighbors
- Points in sparse regions (low density) = potential anomalies
- LOF score > 1 = anomaly, LOF score ≈ 1 = normal

---

## 🚀 Quick Start Instructions

1. **Run Cell 1** (installation) - Fixes numpy compatibility
2. **Restart Runtime** - Click 'Runtime' → 'Restart runtime'
3. **Skip Cell 1** after restart
4. **Run from Cell 2 onwards** - Execute cells sequentially

## 1. Install Required Packages

In [ ]:
# Fix numpy compatibility issue and install required packages
!pip uninstall -y numpy
!pip install numpy==1.26.4
!pip install -q --upgrade scikit-learn pandas matplotlib seaborn plotly

print("✅ Installation complete!")
print("⚠️  Please click 'Runtime' → 'Restart runtime' before proceeding.")
print("    After restart, skip this cell and start from Cell 2.")

## 2. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler
import warnings

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
warnings.filterwarnings('ignore')

# Check versions
import sklearn
print(f"scikit-learn version: {sklearn.__version__}")
print(f"pandas version: {pd.__version__}")
print("✅ All libraries imported successfully!")

## 3. Load Your Google Search Console Data

**Instructions:**
1. Export your GSC data as CSV (Date, Clicks, Impressions, CTR, Position)
2. Run the cell below to upload your file

In [ ]:
from google.colab import files

# Upload your CSV file
print("Please upload your Google Search Console CSV file:")
uploaded = files.upload()

# Get the filename
filename = list(uploaded.keys())[0]
print(f"\n✅ File '{filename}' uploaded successfully!")

## 4. Load and Clean Data

In [ ]:
# Load the data
df = pd.read_csv(filename)

print("Raw data loaded. Cleaning and converting...\n")

# Standardize column names (convert to lowercase, remove spaces)
df.columns = df.columns.str.lower().str.strip()

# Convert date column to datetime
date_column = [col for col in df.columns if 'date' in col.lower()][0]
df['date'] = pd.to_datetime(df[date_column])
if date_column != 'date':
    df = df.drop(columns=[date_column])

# Function to clean numeric columns
def clean_numeric(series):
    """Convert strings with commas, percentages to numeric"""
    if series.dtype == 'object':
        # Remove commas and convert
        series = series.astype(str).str.replace(',', '')
        # Handle percentages
        if series.str.contains('%').any():
            series = series.str.rstrip('%').astype(float) / 100
        else:
            series = pd.to_numeric(series, errors='coerce')
    return pd.to_numeric(series, errors='coerce')

# Clean numeric columns
numeric_columns = ['clicks', 'impressions', 'ctr', 'position']
for col in numeric_columns:
    if col in df.columns:
        df[col] = clean_numeric(df[col])

# Remove any rows with missing critical data
df = df.dropna(subset=['date'])

# Sort by date
df = df.sort_values('date').reset_index(drop=True)

# Aggregate by date if there are duplicates
if df['date'].duplicated().any():
    print("⚠️  Found duplicate dates. Aggregating...")
    agg_dict = {col: 'sum' for col in numeric_columns if col in df.columns and col != 'position'}
    if 'position' in df.columns:
        agg_dict['position'] = 'mean'  # Average position
    df = df.groupby('date').agg(agg_dict).reset_index()

# Display basic info
print("✅ Data cleaned and converted!\n")
print("Dataset Shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst few rows:")
print(df.head())

print("\nData types:")
print(df.dtypes)

print("\nBasic statistics:")
print(df.describe())

print(f"\nDate range: {df['date'].min()} to {df['date'].max()}")
print(f"Total days: {len(df)}")

# Check for any remaining issues
if df.isnull().any().any():
    print("\n⚠️  Warning: Found missing values:")
    print(df.isnull().sum())
    print("\nFilling missing values with forward fill...")
    df = df.ffill().bfill()

## 5. Feature Engineering

LOF benefits greatly from well-engineered features that capture temporal patterns and deviations.

In [ ]:
# Configuration - Choose primary metric to analyze
PRIMARY_METRIC = 'clicks'  # Options: 'clicks', 'impressions', 'ctr', 'position'

# Create a copy for feature engineering
df_features = df.copy()

# Extract temporal features
df_features['day_of_week'] = df_features['date'].dt.dayofweek  # 0=Monday, 6=Sunday
df_features['day_of_month'] = df_features['date'].dt.day
df_features['month'] = df_features['date'].dt.month
df_features['quarter'] = df_features['date'].dt.quarter
df_features['is_weekend'] = df_features['day_of_week'].isin([5, 6]).astype(int)
df_features['week_of_year'] = df_features['date'].dt.isocalendar().week

# Calculate rolling statistics (7-day, 14-day, and 30-day windows)
for metric in ['clicks', 'impressions', 'ctr', 'position']:
    if metric in df_features.columns:
        # Rolling means
        df_features[f'{metric}_rolling_mean_7'] = df_features[metric].rolling(window=7, min_periods=1).mean()
        df_features[f'{metric}_rolling_mean_14'] = df_features[metric].rolling(window=14, min_periods=1).mean()
        df_features[f'{metric}_rolling_mean_30'] = df_features[metric].rolling(window=30, min_periods=1).mean()

        # Rolling standard deviations
        df_features[f'{metric}_rolling_std_7'] = df_features[metric].rolling(window=7, min_periods=1).std()
        df_features[f'{metric}_rolling_std_30'] = df_features[metric].rolling(window=30, min_periods=1).std()

        # Deviation from rolling mean (z-score style)
        df_features[f'{metric}_deviation_7'] = (
            (df_features[metric] - df_features[f'{metric}_rolling_mean_7']) /
            (df_features[f'{metric}_rolling_std_7'] + 1e-5)
        )
        df_features[f'{metric}_deviation_30'] = (
            (df_features[metric] - df_features[f'{metric}_rolling_mean_30']) /
            (df_features[f'{metric}_rolling_std_30'] + 1e-5)
        )

        # Percent change from previous day
        df_features[f'{metric}_pct_change'] = df_features[metric].pct_change().fillna(0)

        # Min/Max in rolling window
        df_features[f'{metric}_rolling_min_7'] = df_features[metric].rolling(window=7, min_periods=1).min()
        df_features[f'{metric}_rolling_max_7'] = df_features[metric].rolling(window=7, min_periods=1).max()

        # Distance from min/max
        df_features[f'{metric}_dist_from_min_7'] = df_features[metric] - df_features[f'{metric}_rolling_min_7']
        df_features[f'{metric}_dist_from_max_7'] = df_features[f'{metric}_rolling_max_7'] - df_features[metric]

# Fill any NaN values with forward fill then backward fill
df_features = df_features.ffill().bfill()

print("✅ Feature engineering complete!")
print(f"\nTotal features created: {len(df_features.columns)}")
print(f"\nSample of engineered features:")
feature_cols = [col for col in df_features.columns if col not in ['date', 'clicks', 'impressions', 'ctr', 'position']]
print(feature_cols[:10])

## 6. Visualize Raw Data

In [ ]:
# Create visualizations for each metric
metrics = ['clicks', 'impressions', 'ctr', 'position']
available_metrics = [m for m in metrics if m in df.columns]

fig, axes = plt.subplots(len(available_metrics), 1, figsize=(16, 4*len(available_metrics)))

if len(available_metrics) == 1:
    axes = [axes]

for idx, metric in enumerate(available_metrics):
    axes[idx].plot(df['date'], df[metric], linewidth=1.5, color='#2E86AB')
    axes[idx].set_xlabel('Date', fontsize=11, fontweight='bold')
    axes[idx].set_ylabel(metric.capitalize(), fontsize=11, fontweight='bold')
    axes[idx].set_title(f'Google Search Console - {metric.capitalize()} Over Time', fontsize=13, fontweight='bold')
    axes[idx].grid(True, alpha=0.3)
    axes[idx].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print(f"\n{PRIMARY_METRIC.capitalize()} Statistics:")
print(f"  Mean: {df[PRIMARY_METRIC].mean():.2f}")
print(f"  Std Dev: {df[PRIMARY_METRIC].std():.2f}")
print(f"  Min: {df[PRIMARY_METRIC].min():.2f}")
print(f"  Max: {df[PRIMARY_METRIC].max():.2f}")

## 7. Prepare Features for LOF

In [ ]:
# Select features for the model
# Exclude date and other non-numeric columns
feature_columns = [col for col in df_features.columns if col != 'date']

# Create feature matrix
X = df_features[feature_columns].values

# Standardize features (VERY important for LOF)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("✅ Features prepared for Local Outlier Factor")
print(f"Feature matrix shape: {X_scaled.shape}")
print(f"Number of samples: {X_scaled.shape[0]}")
print(f"Number of features: {X_scaled.shape[1]}")

## 8. Train Local Outlier Factor Model

In [ ]:
# Configuration
N_NEIGHBORS = 20           # Number of neighbors to consider (key parameter)
CONTAMINATION = 0.05       # Expected proportion of anomalies (5%)
# Adjust: 10-30 neighbors typical, lower = more local, higher = more global

# Initialize Local Outlier Factor
lof = LocalOutlierFactor(
    n_neighbors=N_NEIGHBORS,      # Number of neighbors for density estimation
    contamination=CONTAMINATION,   # Expected proportion of outliers
    novelty=False,                 # False = training mode (not for new data)
    n_jobs=-1                      # Use all CPU cores
)

print("Training Local Outlier Factor model...")
print(f"Number of neighbors: {N_NEIGHBORS}")
print(f"Expected anomaly rate: {CONTAMINATION*100}%\n")

# Fit and predict
predictions = lof.fit_predict(X_scaled)

# Get LOF scores (negative outlier factor)
# More negative = more anomalous
lof_scores = lof.negative_outlier_factor_

# Add results to dataframe
df_features['anomaly'] = predictions  # -1 for anomalies, 1 for normal
df_features['lof_score'] = lof_scores
df_features['is_anomaly'] = df_features['anomaly'] == -1

# Calculate interpretable LOF score (higher = more anomalous)
# Convert negative outlier factor to positive anomaly score
df_features['anomaly_score'] = -lof_scores  # Now higher = more anomalous

print("✅ Model trained successfully!")
print(f"\nAnomalies detected: {df_features['is_anomaly'].sum()} out of {len(df_features)} days")
print(f"Anomaly rate: {df_features['is_anomaly'].sum()/len(df_features)*100:.2f}%")

print(f"\nLOF Score Interpretation:")
print(f"  Scores close to 1.0 = Normal (similar density to neighbors)")
print(f"  Scores > 1.0 = Anomaly (lower density than neighbors)")
print(f"  Higher scores = More anomalous")

## 9. Classify Anomaly Types

In [ ]:
# Classify anomalies as positive (high values) or negative (low values)
# Based on the primary metric deviation from rolling mean
deviation_col = f'{PRIMARY_METRIC}_deviation_7'

df_features['anomaly_type'] = 'Normal'
df_features.loc[
    (df_features['is_anomaly']) & (df_features[deviation_col] > 0),
    'anomaly_type'
] = 'Positive Anomaly'
df_features.loc[
    (df_features['is_anomaly']) & (df_features[deviation_col] <= 0),
    'anomaly_type'
] = 'Negative Anomaly'

print("Anomaly Classification:")
print(f"  Positive anomalies (unusually high): {(df_features['anomaly_type'] == 'Positive Anomaly').sum()}")
print(f"  Negative anomalies (unusually low): {(df_features['anomaly_type'] == 'Negative Anomaly').sum()}")
print(f"  Normal days: {(df_features['anomaly_type'] == 'Normal').sum()}")

## 10. Visualize Anomalies with LOF

In [ ]:
# Main visualization - Primary metric with anomalies
fig, ax = plt.subplots(figsize=(18, 8))

# Plot the primary metric
ax.plot(df_features['date'], df_features[PRIMARY_METRIC],
        label='Actual Values', color='black', linewidth=1.5, zorder=2)

# Plot rolling mean
ax.plot(df_features['date'], df_features[f'{PRIMARY_METRIC}_rolling_mean_7'],
        label='7-day Moving Average', color='#2E86AB', linewidth=2, linestyle='--', alpha=0.7, zorder=1)

# Highlight anomalies
positive_anomalies = df_features[df_features['anomaly_type'] == 'Positive Anomaly']
negative_anomalies = df_features[df_features['anomaly_type'] == 'Negative Anomaly']

ax.scatter(positive_anomalies['date'], positive_anomalies[PRIMARY_METRIC],
          color='#06D6A0', s=150, label='Positive Anomalies',
          zorder=5, edgecolors='black', linewidth=2)

ax.scatter(negative_anomalies['date'], negative_anomalies[PRIMARY_METRIC],
          color='#EF476F', s=150, label='Negative Anomalies',
          zorder=5, edgecolors='black', linewidth=2)

ax.set_xlabel('Date', fontsize=13, fontweight='bold')
ax.set_ylabel(PRIMARY_METRIC.capitalize(), fontsize=13, fontweight='bold')
ax.set_title(f'Anomaly Detection: {PRIMARY_METRIC.capitalize()} (Local Outlier Factor)',
            fontsize=16, fontweight='bold', pad=20)
ax.legend(loc='best', fontsize=11, framealpha=0.9)
ax.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 11. LOF Score Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: LOF scores over time
axes[0].plot(df_features['date'], df_features['anomaly_score'], linewidth=1, color='#A23B72', alpha=0.7)
axes[0].scatter(positive_anomalies['date'], positive_anomalies['anomaly_score'],
               color='#06D6A0', s=100, zorder=5, edgecolors='black', label='Positive')
axes[0].scatter(negative_anomalies['date'], negative_anomalies['anomaly_score'],
               color='#EF476F', s=100, zorder=5, edgecolors='black', label='Negative')
axes[0].axhline(y=1.0, color='orange', linestyle='--', linewidth=2, label='Normal Threshold (1.0)')
axes[0].set_xlabel('Date', fontsize=12, fontweight='bold')
axes[0].set_ylabel('LOF Anomaly Score', fontsize=12, fontweight='bold')
axes[0].set_title('LOF Scores Over Time\n(Higher = More Anomalous)', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)
axes[0].tick_params(axis='x', rotation=45)

# Plot 2: Distribution of LOF scores
normal_scores = df_features[df_features['is_anomaly']==False]['anomaly_score']
anomaly_scores = df_features[df_features['is_anomaly']==True]['anomaly_score']

axes[1].hist(normal_scores, bins=50, alpha=0.7, label='Normal', color='#2E86AB', edgecolor='black')
axes[1].hist(anomaly_scores, bins=20, alpha=0.7, label='Anomalies', color='#EF476F', edgecolor='black')
axes[1].axvline(x=1.0, color='orange', linestyle='--', linewidth=2, label='Normal Threshold')
axes[1].set_xlabel('LOF Anomaly Score', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Frequency', fontsize=12, fontweight='bold')
axes[1].set_title('Distribution of LOF Scores', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print(f"\nLOF Score Statistics:")
print(f"  Mean: {df_features['anomaly_score'].mean():.4f}")
print(f"  Std Dev: {df_features['anomaly_score'].std():.4f}")
print(f"  Min (most normal): {df_features['anomaly_score'].min():.4f}")
print(f"  Max (most anomalous): {df_features['anomaly_score'].max():.4f}")
print(f"\nAnomaly Threshold: ~1.0 (scores > 1.0 are anomalies)")

## 12. Analyze Top Anomalies

In [ ]:
# Get top anomalies sorted by LOF score
anomaly_df = df_features[df_features['is_anomaly']].copy()
anomaly_df = anomaly_df.sort_values('anomaly_score', ascending=False)

print(f"\n{'='*100}")
print(f"TOP 20 ANOMALIES (by LOF Score)")
print(f"{'='*100}\n")

# Select relevant columns for display
display_cols = ['date', 'clicks', 'impressions', 'ctr', 'position', 'anomaly_score', 'anomaly_type']
available_display_cols = [col for col in display_cols if col in anomaly_df.columns]

top_anomalies = anomaly_df.head(20)[available_display_cols].copy()

# Format the display
top_anomalies['date'] = top_anomalies['date'].dt.strftime('%Y-%m-%d')
top_anomalies['anomaly_score'] = top_anomalies['anomaly_score'].apply(lambda x: f"{x:.4f}")

# Rename columns for better display
column_names = {
    'date': 'Date',
    'clicks': 'Clicks',
    'impressions': 'Impressions',
    'ctr': 'CTR',
    'position': 'Position',
    'anomaly_score': 'LOF Score',
    'anomaly_type': 'Type'
}
top_anomalies = top_anomalies.rename(columns=column_names)

print(top_anomalies.to_string(index=False))

# Summary statistics
print(f"\n{'='*100}")
print(f"ANOMALY SUMMARY")
print(f"{'='*100}")
print(f"Total anomalies detected: {len(anomaly_df)} ({len(anomaly_df)/len(df_features)*100:.2f}% of all days)")
print(f"Positive anomalies: {(anomaly_df['anomaly_type'] == 'Positive Anomaly').sum()}")
print(f"Negative anomalies: {(anomaly_df['anomaly_type'] == 'Negative Anomaly').sum()}")
print(f"\nMost anomalous day: {anomaly_df.iloc[0]['date'].strftime('%Y-%m-%d')} (LOF score: {anomaly_df.iloc[0]['anomaly_score']:.4f})")

# Day of week analysis
print(f"\n{'='*100}")
print(f"ANOMALIES BY DAY OF WEEK")
print(f"{'='*100}")
day_names = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
for day_idx, day_name in enumerate(day_names):
    count = (anomaly_df['day_of_week'] == day_idx).sum()
    pct = (count / len(anomaly_df) * 100) if len(anomaly_df) > 0 else 0
    print(f"{day_name:12s}: {count:3d} anomalies ({pct:5.1f}%)")

## 13. Local Density Visualization

This shows why LOF is called "Local" - it considers the density of points in the neighborhood.

In [ ]:
# Visualize the local density concept using 2 key features
from sklearn.decomposition import PCA

# Reduce to 2D for visualization
pca = PCA(n_components=2)
X_2d = pca.fit_transform(X_scaled)

# Create scatter plot colored by LOF score
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: All points colored by LOF score
scatter1 = axes[0].scatter(X_2d[:, 0], X_2d[:, 1],
                          c=df_features['anomaly_score'],
                          cmap='RdYlGn_r', s=60, alpha=0.6, edgecolors='black', linewidth=0.5)
axes[0].set_xlabel('First Principal Component', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Second Principal Component', fontsize=12, fontweight='bold')
axes[0].set_title('LOF Scores in 2D Feature Space\n(Warmer colors = More anomalous)',
                 fontsize=13, fontweight='bold')
plt.colorbar(scatter1, ax=axes[0], label='LOF Anomaly Score')
axes[0].grid(True, alpha=0.3)

# Plot 2: Highlight only anomalies
normal_mask = ~df_features['is_anomaly'].values
anomaly_mask = df_features['is_anomaly'].values

axes[1].scatter(X_2d[normal_mask, 0], X_2d[normal_mask, 1],
               c='lightblue', s=40, alpha=0.4, label='Normal', edgecolors='none')
axes[1].scatter(X_2d[anomaly_mask, 0], X_2d[anomaly_mask, 1],
               c='red', s=100, alpha=0.8, label='Anomalies', edgecolors='black', linewidth=1.5)
axes[1].set_xlabel('First Principal Component', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Second Principal Component', fontsize=12, fontweight='bold')
axes[1].set_title('Anomalies in 2D Feature Space\n(Red points have low local density)',
                 fontsize=13, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 LOF Insight:")
print("Red points are in regions with LOWER DENSITY than their neighbors.")
print("This is why they're flagged as anomalies - they don't fit the local pattern.")

## 14. Compare with Simple Statistical Method

In [ ]:
# Let's compare LOF with a simple Z-score method
from scipy import stats

# Calculate Z-scores for the primary metric
z_scores = np.abs(stats.zscore(df_features[PRIMARY_METRIC]))
z_anomalies = z_scores > 3  # Standard 3-sigma rule

# Compare results
lof_anomalies = df_features['is_anomaly'].values

# Venn diagram data
only_lof = np.sum(lof_anomalies & ~z_anomalies)
only_zscore = np.sum(z_anomalies & ~lof_anomalies)
both = np.sum(lof_anomalies & z_anomalies)

print("Comparison: LOF vs Z-Score Method")
print("="*50)
print(f"LOF only: {only_lof} anomalies")
print(f"Z-score only: {only_zscore} anomalies")
print(f"Both methods: {both} anomalies")
print(f"\nTotal LOF anomalies: {np.sum(lof_anomalies)}")
print(f"Total Z-score anomalies: {np.sum(z_anomalies)}")

print(f"\n💡 Insight:")
print(f"LOF found {only_lof} anomalies that Z-score missed.")
print(f"These are likely SUBTLE or LOCAL anomalies that LOF excels at detecting!")

# Visualize comparison
fig, ax = plt.subplots(figsize=(16, 6))

ax.plot(df_features['date'], df_features[PRIMARY_METRIC],
        label='Actual Values', color='black', linewidth=1.5, alpha=0.5)

# Mark LOF-only anomalies
lof_only_df = df_features[lof_anomalies & ~z_anomalies]
ax.scatter(lof_only_df['date'], lof_only_df[PRIMARY_METRIC],
          color='#06D6A0', s=150, label='LOF Only', zorder=5, edgecolors='black', linewidth=2)

# Mark Z-score-only anomalies
zscore_only_df = df_features[z_anomalies & ~lof_anomalies]
ax.scatter(zscore_only_df['date'], zscore_only_df[PRIMARY_METRIC],
          color='#FFB703', s=150, label='Z-Score Only', zorder=5, edgecolors='black', linewidth=2)

# Mark both
both_df = df_features[lof_anomalies & z_anomalies]
ax.scatter(both_df['date'], both_df[PRIMARY_METRIC],
          color='#EF476F', s=150, label='Both Methods', zorder=5, edgecolors='black', linewidth=2)

ax.set_xlabel('Date', fontsize=13, fontweight='bold')
ax.set_ylabel(PRIMARY_METRIC.capitalize(), fontsize=13, fontweight='bold')
ax.set_title('LOF vs Z-Score: Which Anomalies Does Each Method Find?', fontsize=16, fontweight='bold', pad=20)
ax.legend(loc='best', fontsize=11, framealpha=0.9)
ax.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 15. Export Results

In [ ]:
# Prepare export dataframe with key information
export_cols = ['date', 'clicks', 'impressions', 'ctr', 'position',
               'lof_score', 'anomaly_score', 'is_anomaly', 'anomaly_type',
               'day_of_week', 'is_weekend']
available_export_cols = [col for col in export_cols if col in df_features.columns]

export_df = df_features[available_export_cols].copy()

# Add day name
export_df['day_name'] = export_df['date'].dt.day_name()

# Save to CSV
output_filename = f'gsc_anomaly_detection_lof_{datetime.now().strftime("%Y%m%d_%H%M%S")}.csv'
export_df.to_csv(output_filename, index=False)

print(f"✅ Results exported to: {output_filename}")
print(f"\nDownloading file...")

# Download the file
files.download(output_filename)

print(f"\n📊 Export complete! The file contains:")
print(f"  - All dates with metrics")
print(f"  - LOF scores (higher = more anomalous)")
print(f"  - Anomaly flags and classifications")
print(f"  - Temporal information")

# Also export just the anomalies
anomalies_filename = f'gsc_anomalies_only_lof_{datetime.now().strftime("%Y%m%d_%H%M%S")}.csv'
export_df[export_df['is_anomaly']].to_csv(anomalies_filename, index=False)
files.download(anomalies_filename)
print(f"\n✅ Anomalies-only file exported to: {anomalies_filename}")

## 16. Optional: Parameter Tuning

Adjust sensitivity by changing number of neighbors or contamination.

In [ ]:
# Test different parameter combinations
neighbor_values = [10, 15, 20, 25, 30]
contamination_values = [0.01, 0.03, 0.05, 0.10]

print("Testing different parameter combinations...\n")
print(f"{'N Neighbors':<15} {'Contamination':<15} {'Anomalies':<15} {'Percentage':<15}")
print("="*60)

for n_neigh in neighbor_values:
    for cont in contamination_values:
        model = LocalOutlierFactor(n_neighbors=n_neigh, contamination=cont, n_jobs=-1)
        preds = model.fit_predict(X_scaled)
        n_anomalies = (preds == -1).sum()
        percentage = (n_anomalies / len(preds)) * 100

        marker = " ← Current" if n_neigh == N_NEIGHBORS and cont == CONTAMINATION else ""
        print(f"{n_neigh:<15} {cont:<15.2f} {n_anomalies:<15} {percentage:<15.2f}%{marker}")

print("\n💡 Parameter Guide:")
print("  N_NEIGHBORS:")
    print("    - Lower (10-15): More sensitive to local patterns, more anomalies")
print("    - Higher (25-30): Considers broader context, fewer anomalies")
print("  CONTAMINATION:")
print("    - Lower (0.01): Stricter, fewer anomalies")
print("    - Higher (0.10): More lenient, more anomalies")
print("\nTo change parameters, modify N_NEIGHBORS and CONTAMINATION in Cell 8 and re-run from there.")

## Key Insights: LOF vs Other Methods

### Why LOF is Special

1. **Subtle Anomalies**: Finds points that are normal globally but abnormal locally
2. **No Distribution Assumptions**: Doesn't assume your data follows any particular distribution
3. **Multi-dimensional**: Considers all features simultaneously

### When to Use LOF

✅ **Use LOF when:**
- You have clusters of normal behavior
- Anomalies might be subtle or localized
- Your data has varying density patterns
- You want to detect contextual anomalies

❌ **Consider alternatives when:**
- You have very large datasets (LOF is slower)
- You need real-time detection (use Isolation Forest)
- Anomalies are clearly extreme outliers (Z-score might be enough)

### Interpreting LOF Scores

- **Score ≈ 1.0**: Normal point (similar density to neighbors)
- **Score > 1.0**: Potential anomaly (lower density than neighbors)
- **Score >> 1.0**: Strong anomaly (much lower density)

### Tips for GSC Data

1. **Start with n_neighbors=20**: Good balance for daily data
2. **Adjust based on results**: More false positives? Increase neighbors
3. **Cross-reference**: Compare LOF results with known events
4. **Feature engineering matters**: LOF benefits from good features

Happy anomaly hunting!